# Multi-Region Cell Segmentation Pipeline
# Sequential Processing: Scripts 6a → 6b → 6c

## Overview
This pipeline performs advanced multi-region segmentation of FRET cells by identifying different cellular compartments based on brightness patterns and vesicle detection. The analysis separates cells into multiple functional regions to study heterogeneous protein distributions and dynamics.

## Biological Background
Membrone proteins often show:
- **Brightness heterogeneity**: Different molecular aggregation states
- **Mobile vesicles**: Fast-moving organelles in red-delay channel  
- **Compartmentalization**: Distinct regions with different protein concentrations

## Pipeline Structure
1. **Script 6a**: Export red-delay time series for vesicle detection
2. **Script 6b**: Combine probability maps from external analysis (Ilastik)
3. **Script 6c**: Generate multiple segmentation masks using two approaches

## Input/Output Summary
- **Input**: `*.ptu` files, N&B analysis results, Ilastik probability maps
- **Output**: Multiple region masks (`*_vesicles.tif`, `*_low.tif`, `*_high.tif`, `*_lowNB.tif`, `*_highNB.tif`)

## External Dependencies
- **Ilastik software**: For vesicle detection using "cell counting" approach
- **tttrlib**: For TTTR data processing

---


## SCRIPT 6B: COMBINE PROBABILITY MAPS  


## Probability Map Merging Pipeline

### Overview
After Ilastik analysis, this script combines individual frame probability maps back into time-series TIFF stacks. Ilastik processes individual frames and outputs probability maps for vesicle detection.

### External Processing Step (Manual)
**IMPORTANT**: Between scripts 8a and 8b, process the exported frames in Ilastik:
1. Load individual `*_red_delay_XXX.tif` frames into Ilastik
2. Use "Object Classification" or "Pixel Classification" workflow
3. Train classifier to detect bright, circular vesicles
4. Export probability maps with "_Probabilities.tif" suffix
5. Place results in the same `./Cells/Red_delay/` folder

### Input/Output
- **Input**: `*_red_delay_XXX_Probabilities.tif` (individual probability frames from Ilastik)
- **Output**: `*_red_delay_Probabilities.tif` (merged time-series) in `./Cells/Red_delay_merged/`

In [1]:
# Import required libraries
import os
import glob
import tifffile as tiff
import numpy as np

# Define input and output directories
input_dir = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/Red delay/'       # Folder with individual probability frames
output_dir = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/Merged/'   # Output folder for merged time series
os.makedirs(output_dir, exist_ok=True)     # Create output directory if it doesn’t exist

# Group files by base filename (remove frame indices)
file_groups = {}
for filepath in glob.glob(os.path.join(input_dir, "*_Probabilities.tif")):
    # Extract base name by removing frame index (last two underscore-separated parts)
    base_name = "_".join(os.path.basename(filepath).split("_")[:-2])
    file_groups.setdefault(base_name, []).append(filepath)

# Process each group of time points
for base_name, file_list in file_groups.items():
    # Sort files by frame index to maintain temporal order
    file_list.sort(key=lambda x: int(x.split("_")[-2]))  # Extract frame number
    
    # Load all frames and stack into 3D array (time, height, width)
    img_stack = [tiff.imread(file) for file in file_list]
    img_stack = np.stack(img_stack, axis=0)  # Convert list to 3D NumPy array

    # Save as ImageJ-compatible multi-frame TIFF
    output_path = os.path.join(output_dir, f"{base_name}_Probabilities.tif")
    tiff.imwrite(output_path, img_stack.astype('uint16'), imagej=True)

    print(f"Saved merged file: {output_path}")

print(f"All probability maps merged into {output_dir}")

Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_10_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_12_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_14_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_16_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_2_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_4_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_6_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-1_8_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-5_18_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-5_20_red_delay_Probabilities.tif
Saved merged file: ./Cells/Red_delay_merged/high_FRET_1-5_22_red_delay_Probabilities.tif
Saved merged file: ./Cell